# Learning module 2 — From human input to Appium execution

**Goal:** understand how our Interstellar scenario becomes real UI interactions, and where application knowledge comes from.

**Appium receives automation commands. Our agent interprets the English scenario and decides what to try next.** The application then responds, and tools collect evidence of that response.

This module is a proposed execution design and a learning walkthrough. The identifiers, observations, session references, and HTTP examples below are **illustrations, not results collected from Movies**. Our installed app's actual semantics and driver behavior still need to be inspected. These Markdown lessons do not install Appium or execute the app.

Read one section at a time:

1. Who does what.
2. Configure a session.
3. Interpret the request.
4. Inspect the current UI and its identifiers.
5. Turn one observed target into a command.
6. Complete the scenario and verify evidence.
7. Handle waits, ambiguity, and failure.
8. Decide whether to build an app graph first.
9. Retain useful knowledge and choose the first practical exercise.
10. Compare the Dart/Flutter MCP route with Appium.

## This notebook

Part **02 — Appium, semantics, navigation, and Dart MCP** of the learning sequence. [Notebook index](README.md).

Select the project's `.venv` Python kernel and run this notebook from top to bottom. Its Python setup is included locally; running another notebook in another kernel does not populate this one's variables or PATH. The supported working directories are the project root, `server/`, and `server/sample_scenarios/`.

**Execution scope:** Concepts only.

Saved lesson outputs and embedded screenshots are historical snapshots preserved during the split on November 3, 2026. Newly included setup cells have no saved outputs. Rerunning an executable scenario creates fresh local artifacts; these continue living in `server/.tools/`, not beside the notebooks.


## 8. Who does what?

| Component | Responsibility in our proposed system |
| --- | --- |
| Human | Describe the behavior to test, expected outcome, and any limits; answer essential questions. |
| JEV intent classifier | Recognize the requested operation, such as `run_test`. It does not identify UI controls. |
| Executor agent / LLM | Interpret the scenario, use app context, inspect observations, and propose the next permitted action. |
| MCP tools and adapter | Expose controlled capabilities such as inspect, find, tap, type, and screenshot; validate requests and return observations/errors. Exact tool schemas are a later design step. |
| Appium Python client | Translate adapter calls into automation requests to the Appium server. |
| Appium server + XCUITest driver | Manage the iOS automation session and route supported requests to the platform automation implementation. |
| WebDriverAgent / XCTest | Provide the iOS-side mechanisms that inspect and interact with the application. |
| Movies + real TMDB integration | Render the UI and process its normal application/network behavior. |

For our native iOS path, the command chain is:

```text
Executor agent
    ↓ permitted tool request
MCP adapter → Appium Python client
    ↓ HTTP automation request
Appium server → XCUITest driver
    ↓
WebDriverAgent / XCTest → running iOS app
```

Responses travel back through the automation layers. The LLM receives the resulting observation through our tool interface. Appium's documented architecture is client/server; XCUITest uses WebDriverAgent and Apple's testing framework for native automation. [Appium clients](https://appium.io/docs/en/latest/intro/clients/), [XCUITest architecture](https://appium.github.io/appium-xcuitest-driver/latest/overview/)

The **MCP adapter is part of our design**, not a built-in requirement of Appium. A normal Python test can use Appium directly. We add MCP to give our agent controlled access to those capabilities.

## 9. Configure a session before asking it to control Movies

An Appium **session** is the connection/context for automating a selected target. Starting a session returns a session ID used for subsequent requests. Session capabilities describe the platform and driver; they do not describe the movie scenario or its selectors. [Session capabilities](https://appium.io/docs/en/latest/guides/caps/)

| Profile information | Purpose | Example or unresolved value |
| --- | --- | --- |
| Appium server address | Where the client sends requests; this is connection configuration, not a desired capability | A local server address, to be verified. |
| `platformName` | Select the target platform | `iOS`. |
| `appium:automationName` | Select the automation driver | `XCUITest`. |
| `appium:udid` | Identify the intended device/Simulator | Actual selected Simulator identifier, not yet configured here. |
| `appium:bundleId` or `appium:app` | Identify an installed app or installable app artifact | Actual Movies bundle ID/artifact, to be verified. |
| Backend environment and verification method | Establish that the selected app uses its real backend | Real TMDB integration; keep this in our run profile/preflight, not as an invented Appium capability. |
| Starting-state and action policy | Define permitted lifecycle changes and actions | Search/view only for this scenario; setup/reset behavior must be explicit. |

Appium extension capabilities use the `appium:` namespace. The XCUITest driver documents its app-selection capabilities. [Appium capability rules](https://appium.io/docs/en/latest/guides/caps/), [XCUITest capabilities](https://appium.github.io/appium-xcuitest-driver/latest/reference/capabilities/)

The server, driver, Xcode/Simulator, app installation, and WebDriverAgent path must work before our agent can execute a scenario. Connecting successfully does not itself prove the business scenario passes. We will verify this setup in a later practical lesson.

## 10. Interpret the English request without inventing a UI route

Our input is:

> In Movies, search for Interstellar, open the 2014 movie, and verify that its details show the title Interstellar and release year 2014.

The LLM can extract these facts from the sentence:

| Meaning | Extracted information |
| --- | --- |
| Requested tasks | Search for the movie; open the matching movie's details. |
| Target identity | Interstellar, released in 2014. |
| Expected outcomes | Details title is Interstellar; displayed release information establishes year 2014. |

Structured output supplies a defined shape for those facts. Validation checks its types and constraints; additional checks must preserve the human's meaning. The current notebook dictionary is hand-built to make this interpretation visible. [LangChain structured output](https://docs.langchain.com/oss/python/langchain/structured-output)

The sentence does not tell us whether search is a tab, icon, menu option, or field already on screen. The executor gets that context from the selected app profile, relevant approved documentation/procedures, and **current tool observations**.

At each decision, supply the model with the task and assertions, relevant app knowledge, recent actions, and the current UI snapshot. A screenshot can supplement the hierarchy when the selected model/tool supports vision. The model proposes a supported next action; our adapter/runtime checks the target, permissions, and budget before executing it.

## 11. Inspect the present elements and understand semantics

For native iOS testing, request the current UI source and a screenshot through the adapter. The source describes the platform-exposed automation/accessibility hierarchy. It is not the Flutter widget tree or a list of all routes in the app. Appium exposes source and screenshot commands. [WebDriver API](https://appium.io/docs/en/latest/reference/api/webdriver/)

A **hypothetical** inspection might expose:

| Element type | `name` | `label` | Meaning to the executor |
| --- | --- | --- | --- |
| `XCUIElementTypeButton` | `navigation.search` | Search | A candidate control for reaching search. |
| `XCUIElementTypeTextField` | `search.query` | Search movies | A candidate field for entering the query, when present. |

These are illustrative identifiers, not verified Movies selectors. XCUITest exposes attributes such as type, name, label, value, enabled, visibility, and bounds. Its `name` can come from an identifier or label; other interaction properties may require separate queries. [Element attributes](https://appium.github.io/appium-xcuitest-driver/latest/reference/element-attributes/)

For Flutter, a developer-assigned semantics `identifier` maps to iOS `accessibilityIdentifier`, Android `resource-id`, and a web semantics attribute. The adapter therefore needs platform-appropriate locator strategies. [Flutter semantics identifier](https://api.flutter.dev/flutter/semantics/SemanticsProperties/identifier.html)

| Identifier | What it identifies | Reusable across runs? |
| --- | --- | --- |
| Movie identity: title + year, or a known backend ID | A business entity | Can guide entity selection; still verify current results. |
| Developer semantics/accessibility identifier | A UI element's intended automation identity | A useful versioned hint, subject to current validation. |
| Locator: strategy + value | A query used to find a current element | Reuse only in compatible app/screen context. |
| WebDriver element reference | A specific element found by the current automation session | Treat as temporary, not durable app knowledge. |
| Flutter `ValueKey` | Flutter's widget identity/state association | A key alone does not establish a native XCUITest locator. |

Flutter keys serve widget identity; native semantics exposure is a separate mechanism. [ValueKey documentation](https://api.flutter.dev/flutter/foundation/ValueKey-class.html)

No semantics ID is needed in the English request. Prefer stable IDs when available; otherwise use an unambiguous supported locator. On iOS, XCUITest's `accessibility id` strategy aliases `name`, so even a successful lookup does not by itself prove that a developer-defined ID exists. [Locator strategies](https://appium.github.io/appium-xcuitest-driver/latest/reference/locator-strategies/)

The executor recognizes screen context from observed markers and relevant app knowledge. Appium does not automatically tell it the app's business screen name. Unbuilt lazy-list items and unopened screens may not be available in the current hierarchy; observing one screen is not observing the whole app.

## 12. Trace one action all the way to Appium

Assume observation `OBS-001` shows a unique Search control whose confirmed locator is `accessibility id = navigation.search`. This assumption is only for explaining the protocol.

1. The executor decides that reaching search is the next task-relevant action. If a usable search field is already present, it can skip this navigation step.
2. It requests a permitted tap through our tool interface, referencing the observed target and `OBS-001`.
3. The adapter checks session/app scope, target freshness, uniqueness, actionability, and remaining budget. It resolves the locator using the Appium client.
4. Appium returns an element reference. The client uses that reference for the click request.
5. The adapter obtains a fresh observation and checks whether the intended UI transition occurred. A successful click response alone is not proof of navigation or scenario success.

The following **illustrative HTTP requests are documentation, not executable notebook cells**. The client library ordinarily handles them for us:

```http
POST /session/{sessionId}/element
Content-Type: application/json

{"using": "accessibility id", "value": "navigation.search"}
```

The response supplies an opaque WebDriver element reference. Use that returned reference in the next request:

```http
POST /session/{sessionId}/element/{elementId}/click
Content-Type: application/json

{}
```

After inspecting the search UI and finding its actual input element, text entry uses the input's own reference:

```http
POST /session/{sessionId}/element/{inputElementId}/value
Content-Type: application/json

{"text": "Interstellar"}
```

The session ID, element IDs, and locator have different roles. The LLM must not invent the runtime IDs. Appium's API documents these commands; `accessibility id` is a driver-supported locator strategy. [WebDriver commands](https://appium.io/docs/en/latest/reference/api/webdriver/), [XCUITest locators](https://appium.github.io/appium-xcuitest-driver/latest/reference/locator-strategies/)

The agent selects a constrained tool action; it does not need to generate and execute arbitrary Python code for each tap.

## 13. Walk through the complete Interstellar scenario

This is a **proposed trace**, not a recorded successful execution. Each UI-dependent step must be checked against the actual app.

| Step | Responsible component | Concrete work and required evidence |
| --- | --- | --- |
| 1. Accept and interpret | Classifier, then executor | Recognize `run_test`; preserve the requested title/year assertions. |
| 2. Resolve context | Profile lookup and preflight tools | Select Movies, its installed build, device, driver, real backend, and authorized starting state. Ask for unresolved material information. |
| 3. Create the session | Appium client/server | Start the configured target session and record its identity. |
| 4. Observe | Inspection tools | Capture current source, screenshot, time, and app/session context. Keep an observation reference. |
| 5. Reach search if needed | Executor + guarded tools | Identify current controls, locate the permitted target, act, and observe the resulting UI. |
| 6. Enter the query | Executor + Appium | Find the current input, handle any required clearing/keyboard behavior, enter Interstellar, and wait for the app's search behavior. |
| 7. Identify the movie | Executor + observations | Inspect actual results for the agreed identity. Do not assume the first result is the 2014 movie. If year is not available in the list, opening a candidate to inspect its details may provide identity evidence; verify before accepting it, or clarify when ambiguity remains. |
| 8. Open details | Executor + Appium | Locate the matching current result, open it, and collect details-screen evidence. |
| 9. Evaluate assertions | Analyzer | Compare actual title and release information with Interstellar and 2014 under the recorded rules. Distinguish mismatch from insufficient evidence. |
| 10. Document and deliver | Documenter, then reporter | Preserve the trace, findings, screenshots, missing evidence, and any required failure draft/publication status. |
| 11. Finish lifecycle work | Runtime/tools | Release the automation session according to policy. Keep cleanup problems visible. |

The executor's inner loop is **observe → select one grounded action → validate → execute → observe again**. This repeats while needed and within limits. Its outer handoff remains **Executor → Analyzer → Documenter → Reporter**.

The running Movies app makes its normal TMDB calls. Driving its UI does not automatically prove a fresh API call occurred; preflight or scenario-specific verification must establish any required live-backend claim using available authorized evidence. Existing screenshots and cached content cannot substitute for that evidence.

## 14. Handle waiting, ambiguity, and failure

| Situation | Proposed behavior |
| --- | --- |
| Search results are still loading | Poll for an observable condition with a deadline; retain the last useful observation. |
| A stored element reference is stale after the screen changes | Inspect again and find the current element; preserve the failed attempt. |
| An identifier matches multiple elements | Add supported context or ask when the intended target remains ambiguous. |
| An unexpected screen appears | Inspect its controls and relevant app knowledge; continue only within the scenario and run policy. |
| An essential detail or permission is missing | Use the existing human interrupt/resume mechanism. Re-observe before continuing. |
| The title/year contradicts the requested assertion | Preserve evidence and report a failed assertion. Do not retry merely to erase the failure. |
| The hierarchy cannot establish the required value | Seek supported additional evidence within limits, or return an honest inconclusive/blocked result as appropriate. |

Explicit waits synchronize with conditions rather than assume that a fixed delay is enough. Avoid combining implicit and explicit waits without understanding their interaction. [Selenium waiting strategies](https://www.selenium.dev/documentation/webdriver/waits/)

A tool error is not automatically an application bug. Likewise, failure to find a control may reflect stale context, missing semantics, an unsupported observation method, or actual application behavior. Keep those possibilities separate until evidence supports a conclusion.

Stopping further app actions does not skip the remaining analysis, documentation, and reporting stages. Pending human input is never treated as automatic approval.

## 15. Do we have to build the entire app graph first?

**No. Our first agentic workflow does not require a complete prebuilt app graph or a catalog of every semantics ID.** It does need a configured execution context, enough expected-behavior knowledge, and an automation path that can observe and operate the relevant controls.

Three structures must remain distinct:

| Structure | Represents | How it is obtained |
| --- | --- | --- |
| Current UI hierarchy | Elements exposed in the current automation context | Read from the running app through tools. |
| Application navigation graph | Known screen/state transitions and the conditions/actions that connect them | Optional app documentation, focused onboarding, and accumulated observed transitions. |
| LangGraph StateGraph | Our testing system's classifier, four agent stages, and control flow | Defined by our server design. It is not a map of Movies screens. |

For example, an **illustrative partial navigation graph** might record:

```text
Home
  → Search (empty)
  → Search (results for a query)
  → Movie details (selected movie identity)
```

Once verified, recorded edges represent observations under particular conditions, not guarantees that the route always exists. Search may already be active; a modal or authentication state may change the route. Dynamic data, permissions, feature flags, account state, and app versions make a claim of a complete graph difficult to establish.

By “offline,” distinguish two ideas:

- **A preparation run before ordinary testing:** optional and useful for checking important paths and identifiers. It still uses authorized tools against a running app.
- **Running with networking disabled:** this may expose offline behavior, but it cannot establish how a real API-connected flow currently behaves. Our normal discovery/acceptance runs still use the real backend. Processing previously captured artifacts offline can produce hints, not fresh execution evidence.

A navigation graph also does not tell us what the business behavior should be. Expected outcomes still come from the human scenario and approved application rules.

## 16. Start with focused onboarding and learn useful routes gradually

My recommendation is **a small app profile plus live discovery**, with optional focused onboarding of the first relevant paths. This is a proposal, not a requirement to build a crawler or graph database now.

| Approach | Benefit | Limitation | Fit for this project |
| --- | --- | --- | --- |
| Require a full pre-run crawl | Collects many candidate routes | Cannot promise complete coverage; adds substantial setup and becomes stale | Not a prerequisite. |
| Discover entirely during each scenario | Minimal initial preparation | Repeats exploration and needs good observations/context | Useful starting capability. |
| Small profile + focused walkthrough + reusable verified hints | Checks feasibility early and reduces repeated navigation | Requires versioning and current validation | Recommended progression. |

For the first Movies path:

1. Confirm that the actual installed app and real TMDB integration are the intended target.
2. Inspect the initial UI, search controls, result identity, and details title/year exposure.
3. Add or improve semantics in the app only where actual inspection shows an important control is not reliably observable/actionable.
4. Record the verified route and locator hints with their context; keep discovery within its authorized scope and budget.
5. Run the agent against fresh observations and let it adapt when the UI differs.

Reusable knowledge should retain app/build, platform, locale, relevant account/environment state, screen markers, observed action/transition, locator hints, prerequisites, evidence provenance, and last verification. Represent variable entities as parameters rather than create a permanent screen node for every movie.

Keep **current-run evidence** separate from reusable route hints. When a build/context changes or a locator fails, mark the hint stale or revalidate it; preserve historical evidence. Do not persist session element references or old coordinates as durable identities. A JSON file or simple database can represent early knowledge; a specialized graph store is not required.

Screen knowledge can inform procedural/semantic memory; a run trace can inform episodic memory. RAG can retrieve relevant approved rules. None of them establishes that the current run passes.

## 17. First practical checkpoint for the Appium path

A useful next exercise is to create one Appium session, inspect its UI source, save one screenshot, and identify one real control. Then locate and operate that control through Appium and inspect the new state.

We can use [Appium Inspector](https://appium.github.io/appium-inspector/latest/overview/) to see screenshots and source together and try a locator. It is a graphical Appium client, not a mandatory pre-crawler or a replacement for the runtime observations our agent will use.

For that exercise, the evidence of success is a real tool connection, an observed locator, and a confirmed UI change. It is not yet a completed English-scenario test. We will add its executable cells separately when we reach that milestone.

Before proceeding, explain these distinctions in your own words:

- Which component interprets English, and which executes the device command?
- How does a requested task become a locator grounded in the present UI?
- How do a semantics identifier and a temporary Appium element reference differ?
- Why can we start without knowing every screen in advance?

The same separation supports later Android and browser adapters and a future HeadSpin execution provider. Their actual locators, capabilities, and setup must be verified for each target; an iOS pass does not establish a pass elsewhere.

## 18. Where Dart/Flutter MCP fits

**Dart/Flutter MCP can provide runtime context and, when configured appropriately, execute Flutter UI interactions.** It is another tool integration available to our agents, not another testing agent.

The file you linked is a mixin within the Dart MCP server implementation, not a standalone service to import into Python. Its driver tool supports taps, text entry, scrolling, waits, text reads, and screenshots, with Flutter finders including value keys, text, widget types, and semantics labels. It requires the Flutter driver extension; the source's error guidance calls for enabling that extension before `runApp`, preferably through a separate test entry point. [Linked implementation](https://github.com/dart-lang/ai/blob/main/pkgs/dart_mcp_server/lib/src/mixins/dtd.dart)

Relevant tools include:

| Tool | Potential role in our project |
| --- | --- |
| `dtd` | Discover/connect to the appropriate tooling daemon and registered app. |
| `widget_inspector` | Inspect the current Flutter widget tree through its `get_widget_tree` command. |
| `flutter_driver_command` | Execute supported Flutter-level UI operations. |
| `get_runtime_errors` | Supplement failure analysis with runtime diagnostics. |
| `hot_reload` / `hot_restart` | Development operations; not ordinary actions in our scenario execution policy. |

These are upstream names. Discover the installed server's actual tools and schemas before using them. Some tool categories are configurable. [Tool names](https://github.com/dart-lang/ai/blob/main/pkgs/dart_mcp_server/lib/src/utils/names.dart), [server capabilities and connection guidance](https://github.com/dart-lang/ai/blob/main/pkgs/dart_mcp_server/README.md)

For this local learning project, connecting a compatible debug app and verifying its available extensions would be a separate feasibility step. We have not enabled or connected Dart MCP in the testing server.

## 19. Compare the two runtime paths

The same executor can use different configured adapters. These are alternative command paths; Dart MCP is not an Appium driver inserted between Appium and the app.

```text
Path A — native iOS automation
Executor → MCP adapter → Appium client/server → XCUITest/WDA → app

Path B — Flutter runtime automation
Executor → Dart MCP server → Dart tooling / VM-service connection
         → Flutter inspector or enabled driver extension → app
```

The Dart MCP server exposes tools over standard I/O. Our Python testing system would connect as an MCP client; having tools configured in an IDE does not automatically connect our server's agents. [Official Flutter tooling overview](https://docs.flutter.dev/ai/tools)

| Question | Appium native iOS path | Dart MCP Flutter path |
| --- | --- | --- |
| What context is inspected? | Native automation/accessibility hierarchy | Flutter widget/runtime view, where supported |
| What selects a control? | Native supported locators | Supported Flutter finders |
| Does a Flutter `ValueKey` help directly? | A key alone does not establish a native accessibility identifier | The driver tool supports a value-key finder |
| Does it require a Flutter-specific runtime extension? | Not for this native XCUITest path | Required by the linked Flutter-driver tool |
| Which app technology does this route target? | Native accessibility exposed by supported iOS apps, including Flutter | Compatible Dart/Flutter applications |

The server README describes discovery of Flutter apps registered with DTD during debug/profile runs. Registration does not guarantee every inspector/driver operation is available in every mode or SDK. Verify our chosen build and extensions. [Connection guidance](https://github.com/dart-lang/ai/blob/main/pkgs/dart_mcp_server/README.md)

A native semantics identifier and a Flutter value key remain distinct namespaces. An observed widget tree can help the agent understand a control, but does not by itself prove that Appium can locate that control through native semantics. [Flutter semantics identifiers](https://api.flutter.dev/flutter/semantics/SemanticsProperties/identifier.html)

## 20. Proposed role in our system and the Interstellar example

For our current general-purpose design, keep **Appium as the proposed mobile execution path**, with Dart MCP as an optional Flutter context/diagnostics integration. A Flutter-only development milestone could instead use Dart MCP's driver path after feasibility checks. This lesson does not silently switch the selected backend or promise that either connection is already working.

For example:

1. Resolve the selected app/run profile and execution adapter.
2. If Flutter runtime inspection is available, obtain the current widget view to understand the search control.
3. **Appium path:** inspect native accessibility, resolve a verified native locator, and execute through Appium.
4. **Flutter path:** use a supported finder from the observed Flutter context and execute through the enabled Flutter driver integration.
5. Refresh observations, evaluate the requested title/year, and preserve evidence with its adapter identity.

Use one coordinated action owner for the device; do not let both paths issue competing actions. If a run intentionally combines inspection sources, match the same app instance and record provenance. A failure must not silently change adapters just to produce a pass. Development operations that reload or change the app are separate from the authorized test scenario.

Dart MCP does not provide a complete navigation graph automatically. Its current widget view can help focused onboarding and live grounding; unknown routes and business rules still need observations or approved context. The same answer about offline exploration applies: useful preparation is optional, and current execution still needs fresh evidence.

The real TMDB-backed app can remain the target whichever compatible local adapter is selected. Dart MCP does not automatically provision HeadSpin sessions or configure remote runtime access; cloud reachability and supported setup need separate verification. Our generic agents should therefore depend on defined capabilities rather than require Flutter tooling for every application.

**Pause here:** trace the chosen action path and identify which observations establish widget identity, native locator identity, and the expected-versus-actual result. No new executable cell has been added in this module.

[Index](README.md) · [Previous lesson](01_scenario_input.ipynb) · [Next lesson](03_app_profile.ipynb)